# Análise e Validação do Modelo PINN 

Este notebook tem como objetivo carregar o modelo PINN de Black-Scholes previamente treinado e avaliá-lo em um novo conjunto de dados (PETR4 2024) que não foi visto durante o treinamento (out-of-sample).

**Objetivos:**
1.  **Validar a Acurácia:** Verificar o desempenho de precificação do modelo em dados novos.
2.  **Analisar a Volatilidade:** Extrair a superfície de volatilidade implícita e o *smile* para o novo ativo.
3.  **Verificar as Gregas:** Analisar o comportamento do Delta (∂V/∂S) aprendido.
4.  **Gerar Insights:** Produzir gráficos e métricas de alta qualidade para inclusão em um artigo acadêmico.

In [92]:
# Célula 2: Configuração e Imports
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import TensorDataset, DataLoader
import os
import json

# Importa os módulos do seu projeto (assumindo que o notebook está na raiz)
from src.config import PATHS, DATA_CONFIG, MODEL_CONFIG, VIZ_CONFIG
from src.model import PINN_BlackScholes
from src.physics import black_scholes_residual
from src.data_loader import carregar_taxa_juros
from src.visualization import Visualizer, black_scholes_call_numpy

print(f"PyTorch device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


PyTorch device: cuda


## 1. Carregamento do Modelo e Estatísticas de Treino

Esta é a etapa **mais crítica**. Para que a avaliação funcione, o modelo deve usar **exatamente as mesmas estatísticas de normalização** (`data_stats`) com as quais foi treinado.

O código abaixo tentará carregar o arquivo `data_stats.json` da pasta do modelo. Se falhar, ele tentará carregar o arquivo de dados processado original (`dados_unificados.csv`) e recalcular as estatísticas a partir dele.

In [93]:
# Célula 4: Definir Caminhos
# Define o dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Definir Caminhos ---
# Caminhos para os artefatos do *treinamento original*
history_path = os.path.join(PATHS['results_dir'], 'training_history.csv')
weights_path = os.path.join(PATHS['model_save_dir'], 'best_model_weights.pth')
stats_path = os.path.join(PATHS['model_save_dir'], 'data_stats.json')
processed_data_path = PATHS['processed_data'] # Fallback se stats_path não existir
selic_path = PATHS['selic_data']

# Caminho para os *novos dados* de avaliação
EVAL_DATA_PATH = os.path.join(PATHS['raw_data'], 'PETR4_2024.csv')

print(f"Carregando pesos de: {weights_path}")
print(f"Procurando estatísticas em: {stats_path}")
print(f"Carregando dados de avaliação de: {EVAL_DATA_PATH}")

Carregando pesos de: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\modelo_final\best_model_weights.pth
Procurando estatísticas em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\modelo_final\data_stats.json
Carregando dados de avaliação de: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\dados\brutos\PETR4_2024.csv


In [94]:
# Célula 5: Carregar Estatísticas de Normalização (Obrigatório!)
data_stats = None
try:
    with open(stats_path, 'r') as f:
        data_stats = json.load(f)
    print("✅ Estatísticas de normalização (data_stats.json) carregadas com sucesso.")
except FileNotFoundError:
    print(f"⚠️ Aviso: '{stats_path}' não encontrado.")
    print(f"Tentando recalcular estatísticas do arquivo de dados processados original: {processed_data_path}")
    try:
        df_original = pd.read_csv(processed_data_path)
        data_stats = {
            'S_min': float(df_original['spot_price'].min()),
            'S_max': float(df_original['spot_price'].max()),
            'K_min': float(df_original['strike'].min()),
            'K_max': float(df_original['strike'].max()),
            'T_max': float(df_original['time_to_maturity'].max()),
        }
        print("✅ Estatísticas recalculadas do arquivo de dados original com sucesso.")
        
        # Opcional: Salvar este arquivo para uso futuro
        os.makedirs(os.path.dirname(stats_path), exist_ok=True)
        with open(stats_path, 'w') as f:
            json.dump(data_stats, f, indent=4)
        print(f"Estatísticas salvas em '{stats_path}' para uso futuro.")
        
    except Exception as e:
        print(f"❌ ERRO CRÍTICO: Falha ao carregar ou recalcular estatísticas de treinamento. {e}")
        print("O notebook não pode continuar sem as estatísticas de normalização do treino original.")

if data_stats:
    print("\nEstatísticas de Treinamento:", data_stats)

✅ Estatísticas de normalização (data_stats.json) carregadas com sucesso.

Estatísticas de Treinamento: {'S_min': 1.33, 'S_max': 118.72, 'K_min': 1.15, 'K_max': 148.1, 'T_max': 2.1547619047619047}


In [95]:
# Célula 6: Instanciar e Carregar o Modelo
if 'data_stats' not in locals() or data_stats is None:
    raise RuntimeError("data_stats não foi carregado. Verifique a célula anterior.")

# Instancia o modelo com a mesma configuração e estatísticas do treino
model = PINN_BlackScholes(config=MODEL_CONFIG, data_stats=data_stats)

# Carrega os pesos treinados
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval() # Coloca o modelo em modo de avaliação (desliga dropout, etc)
    print(f"✅ Modelo carregado com sucesso de '{weights_path}' e movido para '{device}'.")
else:
    raise FileNotFoundError(f"❌ ERRO CRÍTICO: Arquivo de pesos não encontrado em '{weights_path}'.")

✅ Modelo carregado com sucesso de 'd:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\modelo_final\best_model_weights.pth' e movido para 'cuda'.


## 2. Carregamento e Preparação dos Dados de Avaliação (PETR4 2024)

Agora carregamos os dados de 2024 e aplicamos **exatamente** o mesmo pré-processamento e normalização usados no treinamento.

In [96]:
# Célula 8: Carregar e Limpar Dados de Avaliação
# Carregar a taxa de juros
df_juros = carregar_taxa_juros(selic_path)

# Carregar novos dados de avaliação
print(f"Carregando e processando dados de '{EVAL_DATA_PATH}'...")
df_eval = pd.read_csv(EVAL_DATA_PATH)

# --- 1. Limpeza e Filtro (similar ao data_loader.py) ---
df_eval = df_eval[df_eval['option_type'] == 'CALL'].dropna(subset=['premium', 'days_to_maturity', 'volatility', 'strike', 'spot_price'])
df_eval = df_eval[(df_eval['premium'] > 0) & 
                  (df_eval['days_to_maturity'] > 0) & 
                  (df_eval['volatility'] > 0) & 
                  (df_eval['strike'] > 0) & 
                  (df_eval['spot_price'] > 0)]

df_eval['time'] = pd.to_datetime(df_eval['time']).dt.date
df_eval['time'] = pd.to_datetime(df_eval['time'])
df_eval['time_to_maturity'] = df_eval['days_to_maturity'] / 252.0

# --- 2. Merge com Juros ---
df_eval = pd.merge_asof(
    df_eval.sort_values('time'),
    df_juros,
    left_on='time',
    right_index=True,
    direction='backward'
).rename(columns={'taxa_anual': 'r'}).dropna(subset=['r'])

print(f"Dados de avaliação limpos. Amostras: {len(df_eval)}")

Carregando dados da taxa de juros de: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\dados\brutos\taxa_selic.csv
Dados da taxa de juros carregados com sucesso.
Carregando e processando dados de 'd:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\dados\brutos\PETR4_2024.csv'...
Dados de avaliação limpos. Amostras: 142620


In [97]:
# Célula 9: Normalização (A ETAPA CORRIGIDA)
# Usamos os 'data_stats' do TREINAMENTO ORIGINAL
df_eval['S_norm'] = (df_eval['spot_price'] - data_stats['S_min']) / (data_stats['S_max'] - data_stats['S_min'])
df_eval['K_norm'] = (df_eval['strike'] - data_stats['K_min']) / (data_stats['K_max'] - data_stats['K_min'])
df_eval['T_norm'] = df_eval['time_to_maturity'] / data_stats['T_max']

# --- Preparar Tensores ---
# A entrada do modelo (problema INVERSE) é [S_norm, K_norm, T_norm, r, premium]
input_features = ['S_norm', 'K_norm', 'T_norm', 'r', 'premium']
X_eval = torch.tensor(df_eval[input_features].values, dtype=torch.float32)
y_eval = torch.tensor(df_eval[['premium']].values, dtype=torch.float32)

print(f"Tensores de avaliação X: {X_eval.shape}, Y: {y_eval.shape}")

Tensores de avaliação X: torch.Size([142620, 5]), Y: torch.Size([142620, 1])


In [98]:
# Célula 10: Criar DataLoader de Avaliação
# Usamos o eval_loader para os plots do Visualizer
eval_dataset = TensorDataset(X_eval, y_eval)
eval_loader = DataLoader(eval_dataset, batch_size=8192, shuffle=False)

print(f"DataLoader de avaliação criado com {len(eval_dataset)} amostras.")

DataLoader de avaliação criado com 142620 amostras.


## 3. Avaliação Quantitativa (Métricas de Erro)

Antes dos gráficos, vamos calcular métricas de erro padrão para ter um resultado quantitativo da performance do modelo nos dados de 2024.

In [99]:
# Célula 12: Gerar Previsões
print("Gerando previsões com o modelo carregado...")
model.eval()
actuals = []
predictions_price = []
predictions_sigma = []

with torch.no_grad():
    for inputs, premiums_real in eval_loader:
        inputs_device = inputs.to(device)
        output = model(inputs_device)
        
        predictions_price.extend(output['price'].cpu().numpy().flatten())
        predictions_sigma.extend(output['sigma'].cpu().numpy().flatten())
        actuals.extend(premiums_real.cpu().numpy().flatten())

price_real_np = np.array(actuals)
price_pred_np = np.array(predictions_price)
sigma_pred_np = np.array(predictions_sigma)

# Armazena resultados em um DataFrame para análise futura
df_eval_results = df_eval.copy()
df_eval_results['price_pred_pinn'] = price_pred_np
df_eval_results['sigma_pred_pinn'] = sigma_pred_np
df_eval_results['price_error'] = price_pred_np - price_real_np
df_eval_results['price_error_pct'] = (price_pred_np - price_real_np) / price_real_np

print("Previsões geradas e salvas em 'df_eval_results'.")

Gerando previsões com o modelo carregado...
Previsões geradas e salvas em 'df_eval_results'.


In [100]:
# Célula 13: Cálculo das Métricas de Erro
mae = mean_absolute_error(price_real_np, price_pred_np)
mse = mean_squared_error(price_real_np, price_pred_np)
rmse = np.sqrt(mse)
r2 = r2_score(price_real_np, price_pred_np)

print("--- Métricas de Avaliação (PETR4 2024) ---")
print(f"R-squared (R²):   {r2:.6f}")
print(f"Mean Absolute Error (MAE):  {mae:.6f}")
print(f"Mean Squared Error (MSE):   {mse:.6f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.6f}")

print("\n--- Estatísticas da Volatilidade Prevista ---")
print(f"Média:   {sigma_pred_np.mean()*100:.2f}%")
print(f"Mínima:  {sigma_pred_np.min()*100:.2f}%")
print(f"Máxima: {sigma_pred_np.max()*100:.2f}%")

--- Métricas de Avaliação (PETR4 2024) ---
R-squared (R²):   0.280070
Mean Absolute Error (MAE):  2.049953
Mean Squared Error (MSE):   20.913593
Root Mean Squared Error (RMSE): 4.573138

--- Estatísticas da Volatilidade Prevista ---
Média:   4.62%
Mínima:  0.00%
Máxima: 88.90%


## 4. Análise Visual e Insights para Artigo

Nesta seção, usamos a classe `Visualizer` do projeto para gerar todos os gráficos de alta qualidade necessários para o artigo.

**Importante:** Passamos o `eval_loader` (com dados de 2024) para o `Visualizer`. Gráficos como `plot_prediction_vs_actual` e `plot_error_by_moneyness` usarão *automaticamente* os novos dados de avaliação.

In [101]:
# Célula 15: Instanciar o Visualizer
# Passamos o 'eval_loader' para que os gráficos de validação usem os dados de 2024
viz = Visualizer(
    model=model, 
    history_path=history_path, 
    val_loader=eval_loader, # <--- USANDO OS DADOS DE 2024
    data_stats=data_stats, 
    config=VIZ_CONFIG
)

Pesos do melhor modelo carregados para visualização.


### Insight 1: Desempenho Histórico do Treinamento

**Pergunta:** O modelo convergiu corretamente durante o treinamento? Os pesos da perda (data vs. pde) se comportaram como esperado?

*(Estes gráficos são baseados no `training_history.csv` e não usam os dados de 2024, mas são essenciais para validar o próprio modelo.)*

In [106]:
# Célula 17: Plotar o histórico de perdas do treinamento original
if VIZ_CONFIG.get('plot_loss_history', False):
    viz.plot_loss_history()

In [107]:
# Célula 18: Plotar a evolução dos pesos da perda
if VIZ_CONFIG.get('plot_weights_history', False):
    viz.plot_weights_history()

Gerando gráfico da evolução dos pesos da perda...
Gráfico de pesos da perda salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\weights_history.png


### Insight 2: Acurácia da Precificação (Out-of-Sample)

**Pergunta:** Quão bem o preço previsto pelo modelo (PINN Price) bate com o preço real de mercado (Market Price) para os dados de 2024?

**Insight para o Artigo:** Este gráfico (juntamente com as métricas $R^2$ e MAE da Seção 3) é a principal evidência de que o modelo generaliza para novos dados. Ele aprendeu a *função* de precificação, não apenas memorizou os dados de treino.

In [108]:
# Célula 20: Plotar Previsão vs. Real
# Este gráfico agora usará o 'eval_loader' (PETR4 2024)
if VIZ_CONFIG.get('plot_price_comparation', False):
    viz.plot_prediction_vs_actual()

Gerando gráfico de Preço Previsto vs. Preço Real...
Gráfico de comparação de preços salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\prediction_vs_actual.png


### Insight 3: Distribuição do Erro por Moneyness

**Pergunta:** O modelo tem algum viés? Ele erra mais para opções "dentro do dinheiro" (In-the-Money) ou "fora do dinheiro" (Out-of-the-Money)?

**Insight para o Artigo:** Um bom modelo deve ter erros distribuídos aleatoriamente em torno de zero (linha vermelha). Se houver um padrão (ex: erros sempre positivos para S/K > 1.1), isso indica um viés sistemático.

In [109]:
# Célula 22: Plotar Erro por Moneyness
# Este gráfico também usará o 'eval_loader' (PETR4 2024)
if VIZ_CONFIG.get('plot_moneyness_comparation', False):
    viz.plot_error_by_moneyness()

Gerando gráfico de Erro por Moneyness...
Gráfico de erro por moneyness salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\error_by_moneyness.png


### Insight 4: A Superfície de Volatilidade Aprendida

**Pergunta:** Qual é a forma da superfície de volatilidade implícita (SVI) que o modelo aprendeu para os dados da PETR4 2024? Ele captura o "sorriso" (smile)?

**Insight para o Artigo:** Esta é a maior vantagem do modelo INVERSE. Em vez de assumir uma volatilidade constante (como o Black-Scholes clássico), o modelo aprende a SVI diretamente dos preços de mercado. Estes gráficos provam que o modelo aprendeu a estrutura de mercado (o *smile* ou *skew*) de forma implícita.

In [110]:
# Célula 24: Plotar Superfície de Volatilidade (3D)
# Este plot gera uma superfície 3D baseada no espaço de inputs
if VIZ_CONFIG.get('plot_vol_surface', False):
    viz.plot_implied_volatility_surface()

Gerando superfície de volatilidade implícita...
Gráfico da superfície de volatilidade salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\implied_volatility_surface.png


In [111]:
# Célula 25: Plotar "Smile" de Volatilidade (2D)
# Este plot mostra um "corte" 2D da superfície
if VIZ_CONFIG.get('plot_vol_smile', False):
    viz.plot_volatility_comparison()

Gerando gráfico de Volatilidade Prevista vs. Volatilidade Real (Histórica)...
Gráfico de curva de volatilidade salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\volatility_smile.png


### Insight 5: As "Greeks" (Derivadas) Aprendidas

**Pergunta:** O modelo aprendeu uma função de precificação suave e diferenciável? O Delta (∂V/∂S) faz sentido?

**Insight para o Artigo:** Como a PINN é forçada a obedecer a EDP, ela aprende uma função suave. Usando `torch.autograd`, podemos calcular as "Greeks" (derivadas) analiticamente, sem necessidade de "bater" o preço. Este gráfico mostra uma superfície de Delta suave e consistente, variando de 0 a 1 como esperado.

In [112]:
# Célula 27: Plotar Superfície do Delta
# Este plot gera a superfície 3D do Delta
if VIZ_CONFIG.get('plot_greeks_comparation', False):
    viz.plot_greeks_surface()

Gerando superfície do Delta (∂V/∂S)...
Gráfico da superfície do Delta salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\delta_surface.png


### Insight 6: Aderência à Física (Resíduo da EDP)

**Pergunta:** O quão bem a solução aprendida pelo modelo (preço + volatilidade) realmente satisfaz a equação de Black-Scholes?

**Insight para o Artigo:** Este mapa de calor mostra o resíduo (erro) da EDP. Valores baixos (próximos de zero) em todo o domínio S-T mostram que o modelo não está apenas *interpolando* os dados (overfitting), mas encontrou uma solução que é *consistente com a física* (a EDP de Black-Scholes).

In [113]:
# Célula 29: Plotar Resíduo da EDP
# Este plot gera o mapa de calor do resíduo da EDP
if VIZ_CONFIG.get('plot_pde_residual', False):
    viz.plot_pde_residual_surface()

Gerando superfície de resíduo da EDP...
Gráfico do resíduo da EDP salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\pde_residual_surface.png


### Insight 7: Comparação de Superfícies de Preço

**Pergunta:** Qual é a aparência da superfície de preço completa aprendida pela PINN?

**Insight para o Artigo:** Este gráfico mostra a superfície de preço 3D (Preço vs. Ativo-S vs. Tempo-T) gerada pelo modelo. A suavidade da superfície é um resultado direto da regularização da EDP. Ela pode ser comparada com a superfície de Black-Scholes analítica (usando uma vol. média) para destacar as diferenças.

In [114]:
# Célula 31: Plotar Comparação de Superfícies de Preço
if VIZ_CONFIG.get('plot_price_surfaces', False): 
    viz.plot_price_surfaces_comparison()

Gerando comparação de superfícies de preço (PINN vs. Analítico)...
Gráfico de comparação de superfícies salvo em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\price_surfaces_comparison.png
